In [2]:
import os
# 1. 경로 설정 
base_path = r'C:\Users\SSAFY\Desktop\SafeDeck_Data'
labels_path = os.path.join(base_path, 'labels') # C:\Users\SSAFY\Desktop\SafeDeck_Data\labels

# def delete_txt_files(path):
#     count = 0
#     # 폴더 안의 모든 파일 확인하기
#     for root, dirs, files in os.walk(path):
#         for file in files:
#             if file.endswith(".txt"):
#                 file_path = os.path.join(root, file)
#                 os.remove(file_path) # 파일 삭제
#                 count += 1

#     return count

# # 실행
# deleted_count delete_txt_files(labels_path)
# print(f"삭제 완료. 총 {deleted_count}개의 .txt 파일이 삭제되었습니다.")



In [ ]:

!pip install git+https://github.com/facebookresearch/segment-anything.git

  Cloning https://github.com/facebookresearch/segment-anything.git to c:\users\ssafy\appdata\local\temp\pip-req-build-dpgrxo2k
  Resolved https://github.com/facebookresearch/segment-anything.git to commit dca509fe793f601edb92606367a655c15ac00fdf
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'


  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/segment-anything.git 'C:\Users\SSAFY\AppData\Local\Temp\pip-req-build-dpgrxo2k'


In [ ]:
import os
import json
import cv2
import numpy as np
from segment_anything import sam_model_registry, SamPredictor

# 경로 및 설정
base_path = r'C:\Users\SSAFY\Desktop\SafeDeck_Data'
sam_checkpoint = os.path.join(base_path, "sam_vit_h_4b8939.pth")
model_type = "vit_h"
device = "cuda" #

# SAM 모델 로드
sam = sam_model_registry[model_type](checkpoint=sam_checkpoint)
sam.to(device=device)
predictor = SamPredictor(sam)

ship_map = {101: 0, 201: 1, 202: 2, 203: 3, 204: 4, 205: 5, 206: 6, 207: 7, 301: 8, 302: 9, 303: 10}

def refine_with_sam(target_split):
    img_dir = os.path.join(base_path, 'images', target_split)
    lbl_dir = os.path.join(base_path, 'labels', target_split)
    
    # JSON 파일 목록 가져오기
    json_files = [f for f in os.listdir(lbl_dir) if f.endswith('.json')]
    
    for j_file in json_files:
        # 이미 txt 파일이 만들어졌는지(이전에 작업을 하다 중단했는지) 확인하는 로직
        txt_name = j_file.replace('.json', '.txt')
        txt_path = os.path.join(lbl_dir, txt_name)

        if os.path.exists(txt_path):
            # 이미 있으면 그냥 넘기기
            continue

        with open(os.path.join(lbl_dir, j_file), 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        img_name = j_file.replace('.json', '.JPG') 
        img_path = os.path.join(img_dir, img_name)
        
        if not os.path.exists(img_path): continue
        
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        predictor.set_image(image)
        
        with open(os.path.join(lbl_dir, j_file.replace('.json', '.txt')), 'w') as f_out:
            for ann in data['annotations']:
                cat_id = ann['category_id']
                if cat_id in ship_map and 'bbox' in ann:
                    class_idx = ship_map[cat_id]
                    # SAM에게 네모 박스 위치 알려주기
                    input_box = np.array(ann['bbox']) 
                    # 박스 좌표 형식 변환: [x1, y1, x2, y2]
                    input_box[2:] = input_box[:2] + input_box[2:] 
                    
                    masks, _, _ = predictor.predict(box=input_box, multimask_output=False)
                    
                    # 마스크(색칠된 부위)를 좌표 점들로 바꾸기
                    contours, _ = cv2.findContours(masks[0].astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                    if contours:
                        points = contours[0].flatten().tolist()
                        normalized_points = []
                        for i in range(0, len(points), 2):
                            normalized_points.append(f"{points[i]/image.shape[1]:.6f} {points[i+1]/image.shape[0]:.6f}")
                        f_out.write(f"{class_idx} {' '.join(normalized_points)}\n")

# 실행
refine_with_sam('train')
refine_with_sam('val')
print(" 세그멘테이션 준비 갈 완료")

c:\Users\SSAFY\miniforge3\envs\safedeck\lib\site-packages\segment_anything\build_sam.py:105: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(f)


✅ SAM이 모든 결함의 테두리를 예쁘게 땄습니다. 세그멘테이션 준비 갈 완료~!


In [ ]:
import os

#  경로 설정 
base_path = r'C:\Users\SSAFY\Desktop\SafeDeck_Data'
labels_path = os.path.join(base_path, 'labels')

def fix_segmentation_labels(path):
    fixed_files = 0
    removed_lines = 0
    
    for root, dirs, files in os.walk(path):
        for file in files:
            if file.endswith(".txt"):
                file_path = os.path.join(root, file)
                with open(file_path, 'r') as f:
                    lines = f.readlines()
                
                # 숫자가 5개 초과인(폴리곤 정보가 있는) 줄만 남기기

                new_lines = [line for line in lines if len(line.split()) > 5]
                
                if len(lines) != len(new_lines):
                    with open(file_path, 'w') as f:
                        f.writelines(new_lines)
                    fixed_files += 1
                    removed_lines += (len(lines) - len(new_lines))
                    
    return fixed_files, removed_lines

# 실행
fixed_f, removed_l = fix_segmentation_labels(labels_path)
print(f"검사 완료 {fixed_f}개의 파일에서 {removed_l}개의 '박스형 데이터'를 제거")

✅ 검사 완료! 244개의 파일에서 245개의 '박스형 데이터'를 제거했습니다. ✨


In [ ]:
from ultralytics import YOLO

#  세그멘테이션 전용 모델 불러오기
model = YOLO('yolov8n-seg.pt') 

# 학습 시작
model.train(
    data='ship_data.yaml', 
    epochs=50,     
    imgsz=640, 
    batch=16, 
    name='SafeDeck_Seg_Final',
    device=0        
)

New https://pypi.org/project/ultralytics/8.4.3 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.253  Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4070 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=ship_data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=SafeDeck_Seg_Final3, nbs=64,

KeyboardInterrupt: 

In [ ]:
from ultralytics import YOLO

# 학습 다시 이어서 할 경우 last 파일 불러오기
model = YOLO(r'C:\Users\SSAFY\Desktop\SafeDeck_Data\runs\segment\SafeDeck_Seg_Final2\weights\last.pt')


model.train(resume=True)

New https://pypi.org/project/ultralytics/8.4.3 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.253  Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4070 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=ship_data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=C:\Users\SSAFY\Desktop\SafeDeck_Data\runs\segment\SafeDeck_Seg_Final2\weights\last.pt, momentum=

ultralytics.utils.metrics.SegmentMetrics object with attributes:

ap_class_index: array([ 2,  3,  4,  5,  6,  8,  9, 10])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x00000254818B9900>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)', 'Precision-Recall(M)', 'F1-Confidence(M)', 'Precision-Confidence(M)', 'Recall-Confidence(M)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.0